In [7]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/raw/train.csv")

# Recreate feature lists (needed since this is a new notebook session)
numerical_features = train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train.select_dtypes(include=['object']).columns.tolist()
numerical_features_only = [c for c in numerical_features if c not in ['Id', 'SalePrice']]

print(train.shape)

(1460, 81)


/var/folders/xd/r1tqvb4s2bvd6gpjtvvbp53r0000gn/T/ipykernel_3942/2813780146.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = train.select_dtypes(include=['object']).columns.tolist()


In [8]:
# Full row duplicates
print("Full duplicate rows:", train.duplicated().sum())

# Duplicate Ids (should never happen — Id is a unique identifier)
print("Duplicate Ids:", train['Id'].duplicated().sum())

Full duplicate rows: 0
Duplicate Ids: 0


In [9]:
# MSSubClass is a coded categorical variable, not a true number
print(train['MSSubClass'].unique())

[ 60  20  70  50 190  45  90 120  30  85  80 160  75 180  40]


In [10]:
# Fix: cast to string/categorical so it's never treated as a continuous number
train['MSSubClass'] = train['MSSubClass'].astype(str)

In [11]:
print(train['MoSold'].unique())
print(train['YrSold'].unique())

[ 2  5  9 12 10  8 11  4  1  7  3  6]
[2008 2007 2006 2009 2010]


In [12]:
# Check for columns with only 1 unique value (truly constant)
constant_cols = [col for col in train.columns if train[col].nunique() <= 1]
print("Constant columns:", constant_cols)

# Check for near-zero variance: one category dominates >99% of rows
for col in categorical_features:
    top_freq = train[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_freq > 0.99:
        print(f"{col}: {top_freq:.4f} dominated by one value")

Constant columns: []
Street: 0.9959 dominated by one value
Utilities: 0.9993 dominated by one value
PoolQC: 0.9952 dominated by one value


In [13]:
# GarageYrBlt should never be in the future or before house was built
print(train[train['GarageYrBlt'] > 2010][['GarageYrBlt', 'YearBuilt', 'YrSold']])

Empty DataFrame
Columns: [GarageYrBlt, YearBuilt, YrSold]
Index: []


In [14]:
# Check YearBuilt / YearRemodAdd sanity
print(train[train['YearBuilt'] > train['YrSold']][['YearBuilt', 'YrSold']])
print(train[train['YearRemodAdd'] < train['YearBuilt']][['YearRemodAdd', 'YearBuilt']])

Empty DataFrame
Columns: [YearBuilt, YrSold]
Index: []
Empty DataFrame
Columns: [YearRemodAdd, YearBuilt]
Index: []


In [15]:
print("Any zero or negative SalePrice?", (train['SalePrice'] <= 0).sum())
print("Any missing SalePrice?", train['SalePrice'].isnull().sum())

Any zero or negative SalePrice? 0
Any missing SalePrice? 0
